In [ ]:
import os
import subprocess

if not os.path.exists("MIAM"):
    subprocess.run(["git", "clone", "https://github.com/zbirobin/MIAM.git"], check=True)
if not os.path.exists("miam-climate-stress"):
    subprocess.run(["git", "clone", "https://github.com/hugoaslm/miam-climate-stress.git"], check=True)

if os.path.basename(os.getcwd()) != "MIAM":
    os.chdir("MIAM")


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q huggingface_hub kagglehub jsonlines seaborn torcheval


In [ ]:
!hf download zbirobin/MIAM geoplant_miam.pt geoplant_dropout.pt geoplant_constant.pt
!mkdir -p models/miam models/dropout models/constant
!mv geoplant_miam.pt models/miam/miam.pt
!mv geoplant_dropout.pt models/dropout/dropout.pt
!mv geoplant_constant.pt models/constant/constant.pt


In [ ]:
import kagglehub

files = [
    "PA_metadata_train.csv",
    "EnvironmentalValues/Climate/Average 1981-2010/PA-train-bioclimatic.csv",
    "EnvironmentalValues/Elevation/PA-train-elevation.csv",
    "EnvironmentalValues/Human Footprint/PA-train-human_footprint.csv",
    "EnvironmentalValues/LandCover/PA-train-landcover.csv",
    "EnvironmentalValues/SoilGrids/PA-train-soilgrids.csv",
    "SateliteTimeSeries-Bioclimatic/values/PA-train-bioclimatic-monthly.csv",
] + [
    f"SateliteTimeSeries-Landsat/values/PA-train-landsat_time_series/"
    f"PA-train-landsat_time_series-{b}.csv"
    for b in ("red", "green", "blue", "nir", "swir1", "swir2")
]

paths = [kagglehub.dataset_download("picekl/geoplant", path=f) for f in files]
geo_path = os.path.dirname(paths[0])


In [ ]:
STRESS_REPO = os.path.join(os.path.dirname(os.getcwd()), "miam-climate-stress")
!mkdir -p ../results ../figures


In [ ]:
!python $STRESS_REPO/scripts/evaluate_climate_stress.py \
    --data_dir $geo_path \
    --checkpoints miam dropout constant \
    --output_dir ../results \
    --batch_size 256 \
    --no_satellite


In [ ]:
!python $STRESS_REPO/scripts/plot_climate_stress.py \
    --results ../results/climate_stress_results.csv \
    --output_dir ../figures \
    --format png


In [ ]:
import pandas as pd

df = pd.read_csv("../results/climate_stress_results.csv")
pivot = df.pivot_table(values="auroc", index="condition", columns="method", aggfunc="first")
for method in pivot.columns:
    if "clean" in pivot.index:
        pivot[f"{method}_robustness"] = pivot[method] / pivot.loc["clean", method]
pivot.round(4).to_csv("../results/climate_stress_summary.csv")
print(pivot.round(4).to_string())


In [ ]:
df = pd.read_csv("../results/climate_stress_results.csv")
for method in df["method"].unique():
    m = df[df["method"] == method]
    clean = m[m["condition"] == "clean"]["auroc"].values[0]
    cm = m[m["condition"] == "climate_missing"]
    drop = clean - cm["auroc"].values[0] if len(cm) else float("nan")
    noise = m[m["condition"].str.contains("noise")]["auroc"]
    line = f"{method}: clean={clean:.4f} climate_missing_drop={drop:+.4f}"
    if len(noise):
        line += f" worst_noise={noise.min():.4f}"
    print(line)


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

rows = []
for method in ["miam", "modality_dropout", "constant"]:
    result_file = f"figures_tables/table_1/results/{method}/all_together.jsonl"
    if os.path.exists(result_file):
        with open(result_file) as f:
            for line in f:
                row = json.loads(line)
                rows.append({
                    "method": method.replace("modality_dropout", "dropout"),
                    "condition": row["variables"],
                    "auroc": row["test_auroc"],
                })

fallback_df = pd.DataFrame(rows)
print(fallback_df.pivot_table(values="auroc", index="condition", columns="method"))

fig, ax = plt.subplots(figsize=(10, 5))
conditions_plot = ["tabular", "timeseries", "climatic_timeseries", "sentinel2_patches"]
colors = {"miam": "#2ecc71", "dropout": "#e74c3c", "constant": "#3498db"}
x = np.arange(len(conditions_plot))
width = 0.25
for i, method in enumerate(["miam", "dropout", "constant"]):
    m = fallback_df[fallback_df["method"] == method]
    all_score = m[m["condition"] == "all"]["auroc"].values[0]
    scores = []
    for cond in conditions_plot:
        r = m[m["condition"] == cond]
        scores.append(all_score - r["auroc"].values[0] if len(r) > 0 else np.nan)
    ax.bar(x + i * width, scores, width, label=method.upper(), color=colors[method])
ax.set_xticks(x + width)
ax.set_xticklabels([c.replace("_", "\n") for c in conditions_plot])
ax.set_ylabel("AUROC drop")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/modality_sensitivity_fallback.png", dpi=150)
plt.show()
